# Control flow, functions, and scope

By this point you already know how to write an `if`, a loop, and a function. The intermediate step is understanding the deeper rules behind them: how names are resolved across scopes, how closures capture variables, how function signatures shape APIs, and how control-flow choices affect readability.

Scope is especially important because many bugs are really name-resolution mistakes. A function can read from an outer scope, shadow a name from outside, or capture a changing value in a closure. Without a clear model of that behaviour, code may look reasonable while doing something surprising.

As you read each notebook, try to explain which scope each name belongs to and why Python resolves it there. That is the foundation for writing reliable functions later.

## Visual model

```text
global scope
  -> function scope
       -> inner function scope
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Loops, and the `else` nobody expects

In [ ]:
for item in collection:      # iterates ANYTHING iterable (Module 14)
    ...

for i, item in enumerate(collection, start=1):
    ...

for a, b in zip(xs, ys, strict=True):    # strict=True is 3.10+ and you want it
    ...

for key, value in mapping.items():
    ...

`zip(strict=True)` raises if the iterables have different lengths. Without it,
`zip` silently stops at the shortest, which has hidden many data bugs. Default
to `strict=True` unless truncation is genuinely intended.

### `for ... else`

The `else` clause runs **if the loop completed without `break`**. It is not "if
the loop body never ran".

In [ ]:
for user in users:
    if user.is_admin:
        print("found an admin")
        break
else:
    print("no admin found")        # runs only if we never broke out

Read `else` here as `nobreak` and it becomes obvious. It exists to remove the
`found = False` flag variable:

In [ ]:
found = False                       # the pattern for ... else replaces
for user in users:
    if user.is_admin:
        found = True
        break
if not found:
    ...

It is rare in real code, and it is on every Python quiz.

### Loop control

```text
break        # exit the innermost loop
continue     # next iteration
```


There is no labelled break. To exit nested loops, either extract the loops into
a function and `return`, or use a flag, or iterate a product:

In [ ]:
from itertools import product
for i, j in product(range(n), range(m)):
    if done(i, j):
        break                       # one loop, so one break is enough

Extracting to a function is almost always the cleanest of the three.

### Do not mutate what you are iterating

In [ ]:
for x in items:
    if pred(x):
        items.remove(x)             # silently skips elements (Module 02, q11)

items = [x for x in items if not pred(x)]      # correct
items[:] = [x for x in items if not pred(x)]   # correct, and in place

---

## Concept 3. `match`: structural pattern matching, not a switch

`match` (3.10+) destructures values. Using it as a C-style switch wastes it.

In [ ]:
match command.split():
    case ["go", direction]:
        move(direction)
    case ["take", *items]:                 # capture the rest
        for item in items:
            take(item)
    case ["quit" | "exit"]:                # alternatives
        raise SystemExit
    case []:
        print("say something")
    case _:                                 # the default; _ matches anything
        print(f"unknown: {command}")

It matches structure, types, and attributes:

In [ ]:
match event:
    case {"type": "click", "pos": (x, y)}:          # dict + tuple shape
        handle_click(x, y)
    case {"type": "key", "code": int() as code}:    # type check + capture
        handle_key(code)
    case Point(x=0, y=0):                            # class patterns
        print("origin")
    case Point(x=x, y=y) if x == y:                  # a guard
        print("diagonal")

Two traps:

**A bare name is a capture, not a comparison.**

```text
case OK:              # binds anything to the name OK. Always matches!
case Status.OK:       # a dotted name IS compared. This is what you meant.
```


This is the number one `match` bug. Any pattern that is a plain identifier
captures; only dotted names, literals, and class patterns compare.

**Class patterns need `__match_args__`** for positional matching, which
`@dataclass` provides automatically (Module 11).

When is `match` worth it? When you are destructuring nested data — parsing,
protocol handling, AST walking, event dispatch. For dispatching on a single
value, a dict of functions is clearer and faster.

---

## Concept 4. Functions: the six kinds of parameter

In [ ]:
def f(pos_only, /, standard, *args, kw_only, **kwargs):
    ...

| Kind | Declared | Called as |
|---|---|---|
| Positional-only | before `/` | `f(1)` only |
| Positional-or-keyword | between `/` and `*` | `f(1)` or `f(standard=1)` |
| Var-positional | `*args` | extra positionals collected into a tuple |
| Keyword-only | after `*` | `f(kw_only=1)` only |
| Var-keyword | `**kwargs` | extra keywords collected into a dict |

In [ ]:
def connect(host, port=5432, /, *, timeout=30, retries=3, **options):
    ...

connect("db", 5432, timeout=10, ssl=True)      # ok
connect(host="db")                              # TypeError: host is positional-only
connect("db", 5432, 10)                         # TypeError: timeout is keyword-only

**Why bother?**

- `/` (positional-only) frees you to rename parameters later without breaking
  callers. The standard library uses it heavily for exactly this reason.
- `*` (keyword-only) forces call sites to be readable. `resize(img, 800, 600,
  True, False)` is unreadable; `resize(img, width=800, height=600,
  preserve_aspect=True, upscale=False)` is not.

**Rule of thumb: any boolean parameter should be keyword-only.** A bare `True`
at a call site carries no information.

### Arguments are unpacked, not copied

In [ ]:
args = (1, 2)
kwargs = {"c": 3}
f(*args, **kwargs)          # equivalent to f(1, 2, c=3)

### The mutable default, again

```text
def f(items=[]):            # WRONG -- evaluated once at def time (Module 02)
def f(items=None):          # right
    if items is None:
        items = []
```


Same for `{}`, `set()`, `datetime.now()`, and any expression whose value should
be per-call. `ruff` rule `B006` catches it.

### Functions are objects

In [ ]:
def greet(name): return f"hi {name}"

greet.__name__          # 'greet'
greet.__doc__           # the docstring
greet.__defaults__      # the default values tuple
greet.__annotations__   # the type hints, as a dict

handlers = {"greet": greet}          # store them
def apply(fn, x): return fn(x)       # pass them
def make(): return greet             # return them

This is what makes decorators, callbacks, and higher-order functions possible,
and it is the subject of Module 15.

---

## Concept 7. Type hints

Hints are not enforced at runtime. They are checked by mypy or pyright, read by
your editor, and used by libraries like Pydantic and FastAPI. Module 17 is the
full treatment; this is the working subset.

In [ ]:
def greet(name: str, times: int = 1) -> str: ...

def parse(raw: str) -> dict[str, int]: ...            # builtin generics, 3.9+
def find(xs: list[int]) -> int | None: ...            # union syntax, 3.10+
def apply(fn: Callable[[int], str], x: int) -> str: ...

from collections.abc import Iterable, Sequence
def total(values: Iterable[float]) -> float: ...      # accept ANY iterable

Two habits worth forming now:

**Accept the widest type, return the narrowest.** Take `Iterable[str]`, not
`list[str]` — then a generator, a tuple, or a set all work. Return `list[str]`,
not `Iterable[str]` — then the caller knows they can index it.

**`X | None` is not optional-as-in-omittable**; it means the value may be `None`.
A parameter is omittable because it has a default.

Write hints on every function you write in this course. Not because Python needs
them, but because writing the return type forces you to decide what the function
actually produces — which is where half of all design bugs are found.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Statements and expressions
- Section 2: Loops, and the `else` nobody expects
- Section 3: `match`: structural pattern matching, not a switch
- Section 4: Functions: the six kinds of parameter
- Section 5: Scope: LEGB
- Section 6: Closures
- Section 7: Type hints

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from typing import Any


# --- 1 -----------------------------------------------------------------------

---

## `resize`

PROBLEM: resize(img, 800, 600, True, False, 90) is unreadable, and any

In [ ]:
def resize(image: Any, w: int, h: int, keep_aspect: bool, upscale: bool,
           quality: int) -> Any:
    """PROBLEM: resize(img, 800, 600, True, False, 90) is unreadable, and any
    caller who swaps two booleans gets silently wrong output.

    TODO: make w and h positional (they have an obvious order), and everything
    else keyword-only. Give sensible defaults.
    """
    return (image, w, h, keep_aspect, upscale, quality)

---

## `add_tag`

PROBLEM: the classic mutable default.

In [ ]:
def add_tag(item: dict[str, Any], tag: str, tags: list[str] = []) -> list[str]:
    """PROBLEM: the classic mutable default.

    TODO: fix it, and make sure an explicitly passed EMPTY list is still used
    rather than replaced.
    """
    tags.append(tag)
    return tags

---

## `fetch`

PROBLEM: ten parameters. Every call site is a wall of keywords, and

In [ ]:
def fetch(url: str, timeout: int = 30, retries: int = 3, backoff: float = 1.5,
          verify_ssl: bool = True, follow_redirects: bool = True,
          max_redirects: int = 10, headers: dict[str, str] | None = None,
          proxy: str | None = None, user_agent: str | None = None) -> str:
    """PROBLEM: ten parameters. Every call site is a wall of keywords, and
    adding an eleventh option means touching this signature again.

    TODO: keep `url` as the only required parameter and group the rest into a
    single options object (a frozen dataclass is ideal -- Module 11 -- but a
    plain class or a TypedDict is acceptable now). Callers should be able to
    build one options object and reuse it.
    """
    return url

---

## `get_user`

PROBLEM: three optional parameters where EXACTLY ONE must be given. The

In [ ]:
def get_user(user_id: int | None = None, email: str | None = None,
             username: str | None = None) -> dict[str, Any]:
    """PROBLEM: three optional parameters where EXACTLY ONE must be given. The
    signature does not say that, so the check has to happen at runtime and the
    type checker cannot help.

    TODO: split into three functions with unambiguous names. Note in a comment
    what you gained and what you lost.
    """
    provided = [p for p in (user_id, email, username) if p is not None]
    if len(provided) != 1:
        raise ValueError("give exactly one of user_id, email, username")
    return {"found": provided[0]}

---

## `parse_date`

PROBLEM: returns a date on success and None on failure, so every caller

In [ ]:
def parse_date(text: str, fmt: str = "%Y-%m-%d") -> Any:
    """PROBLEM: returns a date on success and None on failure, so every caller
    must remember to check, and the type is `date | None` forever.

    TODO: offer BOTH shapes, the way the standard library does:
      parse_date(text)            -> raises ValueError on bad input
      parse_date_or_none(text)    -> returns None on bad input
    Name them so the behaviour is obvious at the call site. (int() and
    dict.get() are the models here.)
    """
    from datetime import datetime
    try:
        return datetime.strptime(text, fmt).date()
    except ValueError:
        return None

---

## `send_email`

PROBLEM: no type hints, no defaults, everything positional and required.

In [ ]:
def send_email(to, subject, body, cc, bcc, reply_to, attachments, priority,
               html):  # type: ignore[no-untyped-def]
    """PROBLEM: no type hints, no defaults, everything positional and required.

    TODO: full hints, sensible defaults, keyword-only for everything after
    `to`, and use Iterable rather than list for the address parameters so
    callers can pass a generator or a tuple.
    """
    return locals()

---

## `test_resize_rejects_positional_booleans`

_test resize rejects positional booleans_

In [ ]:
def test_resize_rejects_positional_booleans() -> None:
    resize(object(), 800, 600, keep_aspect=True)
    try:
        resize(object(), 800, 600, True)  # type: ignore[misc]
    except TypeError:
        return
    raise AssertionError("booleans must be keyword-only")

---

## `test_add_tag_has_no_shared_state`

_test add tag has no shared state_

In [ ]:
def test_add_tag_has_no_shared_state() -> None:
    assert add_tag({}, "a") == ["a"]
    assert add_tag({}, "b") == ["b"], "state leaked between calls"
    given: list[str] = []
    assert add_tag({}, "c", given) is given, "an explicit list must be used"

---

## `test_fetch_takes_options_object`

_test fetch takes options object_

In [ ]:
def test_fetch_takes_options_object() -> None:
    import inspect
    params = inspect.signature(fetch).parameters
    assert len(params) <= 3, f"still {len(params)} parameters"

---

## `test_get_user_is_three_functions`

_test get user is three functions_

In [ ]:
def test_get_user_is_three_functions() -> None:
    assert "get_user_by_id" in globals(), "split get_user into named functions"
    assert "get_user_by_email" in globals()

---

## `test_parse_date_pair`

_test parse date pair_

In [ ]:
def test_parse_date_pair() -> None:
    assert "parse_date_or_none" in globals()
    assert parse_date_or_none("nonsense") is None  # type: ignore[name-defined]  # noqa: F821
    try:
        parse_date("nonsense")
    except ValueError:
        return
    raise AssertionError("parse_date must raise on bad input")

---

## `test_send_email_is_typed`

_test send email is typed_

In [ ]:
def test_send_email_is_typed() -> None:
    import inspect
    sig = inspect.signature(send_email)
    assert all(p.annotation is not inspect.Parameter.empty
               for p in sig.parameters.values()), "add type hints"

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    failed = 0
    for t in tests:
        try:
            t()
            print(f"  PASS  {t.__name__}")
        except Exception as exc:
            failed += 1
            print(f"  FAIL  {t.__name__}: {type(exc).__name__}: {exc}")
    print(f"\n{len(tests) - failed}/{len(tests)} passing")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.